# TrueCarry club/hosel/ball YOLO fine-tune — TWO variants (Colab GPU)
Trains both datasets in one run:
- **all** — every session, junk shots rejected (volume model)
- **jul17** — July 17 range session only, TT+Garmin truth, cleanest capture (purity model)

1. Runtime → Change runtime type → **GPU** (T4 is plenty).
2. Run the first cell, upload `yolo_datasets.zip` (made by `tools/experimental/export_yolo_dataset.py`).
3. Run all. ~30–50 min on T4 for both. Downloads `ClubDetectorV2_all` + `ClubDetectorV2_jul17` mlpackage zips at the end.
4. Send both back — the eval harness picks the winner before anything ships in-app.

In [ ]:
!pip -q install ultralytics coremltools
from google.colab import files
up = files.upload()   # upload yolo_datasets.zip
!unzip -q yolo_datasets.zip -d /content/
!ls /content/yolo_datasets

In [ ]:
from ultralytics import YOLO
# Small model: 360px source frames, 3 classes, ~1-2k images — yolov8n is the
# right size and exports light for on-device. Never mirror: play direction matters.
VARIANTS = ['all', 'jul17']
for name in VARIANTS:
    model = YOLO('yolov8n.pt')
    model.train(data=f'/content/yolo_datasets/{name}/data.yaml', epochs=120, imgsz=640,
                batch=32, patience=25, degrees=5, translate=0.08, scale=0.25,
                fliplr=0.0,
                project='/content/runs', name=f'club_v2_{name}')

In [ ]:
# Validate both on their own held-out shots, and cross-validate the 'all'
# model on jul17's val split (does volume help or hurt the clean session?)
from ultralytics import YOLO
for name in ['all', 'jul17']:
    m = YOLO(f'/content/runs/club_v2_{name}/weights/best.pt')
    r = m.val(data=f'/content/yolo_datasets/{name}/data.yaml')
    print(f'== {name} on own val ==', dict(zip(['head', 'hosel', 'ball'], r.box.maps)))
r = YOLO('/content/runs/club_v2_all/weights/best.pt').val(
    data='/content/yolo_datasets/jul17/data.yaml')
print('== all-model on jul17 val ==', dict(zip(['head', 'hosel', 'ball'], r.box.maps)))

In [ ]:
# Export CoreML for both (same raw-output shape family the app's detector parses)
import shutil
from ultralytics import YOLO
from google.colab import files as gfiles
for name in ['all', 'jul17']:
    YOLO(f'/content/runs/club_v2_{name}/weights/best.pt').export(
        format='coreml', imgsz=640, nms=False)
    shutil.make_archive(f'/content/ClubDetectorV2_{name}.mlpackage', 'zip',
                        f'/content/runs/club_v2_{name}/weights/best.mlpackage')
    gfiles.download(f'/content/ClubDetectorV2_{name}.mlpackage.zip')